# LSTM Classifier Tutorial: Rubik's Cube Solving Method Detection

**Learning Objective**: Build a bidirectional LSTM with attention mechanism to classify Rubik's cube solving methods from state trajectories.

## What You'll Learn

By the end of this, you'll understand how to:
1. Handle variable-length sequence data in PyTorch
2. Implement custom Dataset classes for complex data
3. Build bidirectional LSTMs with attention mechanisms
4. Handle class imbalance with weighted loss functions
5. Visualize attention weights to understand model behavior

## Problem Overview

**Task**: Given a sequence of Rubik's cube states during solving, predict which algorithm/method was used:

- **CFOP** (Cross, F2L, OLL, PLL) - Most popular speedcubing method
- **Roux** - Block-building method with M-slice moves  
- **ZZ** - Edge orientation focused method
- **LBL** (Layer By Layer) - Beginner method, solves one layer at a time
- **Petrus** - Block-building with minimal moves

## Data Characteristics

- **State Representation**: 324-dimensional one-hot encoded vectors (54 stickers × 6 colors)
- **Move Information**: Integer indices (0-53) representing cube moves
- **Sequence Lengths**: Highly variable (27-165 steps) - **Challenge!**
- **Dataset Size**: 2,384 complete solving trajectories

## Tutorial Structure

Each step includes:
- **Theory**: Key concepts explained
- **Code Challenge**: Fill in the missing code
- **Expected Output**: What you should see
- **Hints**: If you get stuck

Let's begin!


In [ ]:
# Step 1: Import Required Libraries

# CODE CHALLENGE: Import all necessary libraries
# Fill in the missing imports below

import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# TODO: Import Dataset and DataLoader from torch.utils.data
from torch.utils.data import _____, _____

# TODO: Import RNN utilities for handling variable-length sequences
from torch.nn.utils.rnn import _____, _____, _____

# TODO: Import train_test_split for data splitting
from sklearn.model_selection import _____

# TODO: Import classification metrics
from sklearn.metrics import _____, _____

# TODO: Import plotting libraries
import _____ as plt
import _____ as sns

# TODO: Import tqdm for progress bars
from _____ import tqdm

import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# EXPECTED OUTPUT:
# Libraries imported successfully!
# PyTorch version: 2.x.x
# CUDA available: True/False
# GPU: [GPU name if available]


Libraries imported successfully!
PyTorch version: 2.3.1+cu121
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB MIG 3g.20gb


## Step 2: Understanding the Data Format

**Theory**: Before building our model, we need to understand our data structure. Each trajectory contains:
- A sequence of cube states (324-dim vectors)
- A sequence of moves (integer indices 0-53)
- The solving method label

**Challenge**: Load and analyze the dataset to understand sequence lengths and method distributions.


In [ ]:
# CODE CHALLENGE: Load and analyze the dataset

data_path = 'dataset_generator/data/all_vector.json'

print("Loading dataset...")
# TODO: Load the JSON data
with open(data_path, 'r') as f:
    data = _____

print(f"Total trajectories: {len(data)}")

# Examine one sample
sample = data[0]
print(f"\nSample structure:")
print(f"Method: {sample['method']}")

# TODO: Print the number of steps in this sample
print(f"Number of steps: {_____}")

# TODO: Print the keys in the first step
print(f"First step keys: {_____}")

# TODO: Print the length of the state vector
print(f"State vector length: {_____}")

# TODO: Print the move index from the first step
print(f"Move index: {_____}")

# Analyze methods and sequence lengths
methods = {}
lengths = []

# TODO: Complete the analysis loop
for solve in data:
    method = solve['_____']  # Get the method
    length = len(solve['_____'])  # Get number of steps
    
    if method not in methods:
        methods[method] = []
    methods[method].append(length)
    lengths.append(length)

print(f"\nMethod distribution:")
for method, method_lengths in methods.items():
    # TODO: Complete the statistics calculation
    print(f"{method}: {len(method_lengths)} solves, "
          f"length range: {_____}-{_____}, "  # min and max
          f"mean: {_____:.1f}")  # mean length

print(f"\nOverall sequence length statistics:")
# TODO: Calculate overall statistics
print(f"Min: {_____}, Max: {_____}, Mean: {_____:.1f}")

# EXPECTED OUTPUT:
# Total trajectories: 2384
# Method: ZZ (or another method)
# Number of steps: ~40-50
# State vector length: 324
# Move index: 0-53
# Method distribution with counts and length ranges


Loading dataset...
Total trajectories: 2384

Sample structure:
Method: ZZ
Number of steps: 46
First step keys: dict_keys(['state', 'move'])
State vector length: 324
Move index: 12

Method distribution:
ZZ: 500 solves, length range: 35-62, mean: 49.0
Roux: 468 solves, length range: 31-58, mean: 44.5
LBL: 500 solves, length range: 70-165, mean: 121.1
CFOP: 500 solves, length range: 30-60, mean: 49.1
Petrus: 416 solves, length range: 27-48, mean: 38.0

Overall sequence length statistics:
Min: 27, Max: 165, Mean: 61.3


## Step 3: Create the Dataset Class

**Theory**: PyTorch's `Dataset` class allows us to create custom data loaders. For sequence data, we need to:
1. Convert lists to tensors
2. Handle variable-length sequences
3. Calculate class weights for imbalanced data
4. Support data augmentation

**Key Challenge**: Variable-length sequences need special handling in PyTorch!


In [ ]:
# CODE CHALLENGE: Implement the CubeTrajectoryDataset class

class CubeTrajectoryDataset(Dataset):
    """
    PyTorch Dataset for cube solving trajectories.
    
    HINT: Remember to inherit from Dataset and implement __len__ and __getitem__
    """
    
    def __init__(self, data_path, max_length=None, augment=False):
        print(f"Loading dataset from {data_path}...")
        
        # TODO: Load the JSON data
        with open(data_path, 'r') as f:
            self.data = _____
        
        # TODO: Create method to label mapping
        # Get unique methods and sort them
        self.methods = sorted(list(set(solve['_____'] for solve in self.data)))
        
        # TODO: Create bidirectional mapping between methods and indices
        self.method_to_idx = {method: _____ for _____, method in enumerate(self.methods)}
        self.idx_to_method = {_____: method for method, _____ in self.method_to_idx.items()}
        
        self.max_length = max_length
        self.augment = augment
        
        # TODO: Calculate class weights for handling imbalance
        method_counts = {}
        for solve in self.data:
            method = solve['method']
            method_counts[method] = method_counts.get(method, 0) + 1
        
        total_samples = len(self.data)
        # Formula: total_samples / (num_classes * samples_in_class)
        self.class_weights = torch.tensor([
            _____ / (len(self.methods) * method_counts[method])
            for method in self.methods
        ], dtype=torch.float32)
        
        print(f"Loaded {len(self.data)} trajectories")
        print(f"Methods: {self.methods}")
        print(f"Class weights: {self.class_weights.numpy()}")
    
    def __len__(self):
        # TODO: Return the number of samples
        return _____
    
    def __getitem__(self, idx):
        """
        HINT: This method should return (states, moves, label, length)
        """
        solve = self.data[idx]
        
        # TODO: Extract states and moves from the solve data
        states = []
        moves = []
        
        for step in solve['_____']:  # What key contains the steps?
            states.append(step['_____'])  # What key contains the state?
            # Handle -1 move indices (end of solve)
            move_idx = step['move'] if step['move'] >= 0 else 0
            moves.append(move_idx)
        
        # TODO: Convert to tensors with appropriate dtypes
        states = torch.tensor(states, dtype=torch._____)  # float32 for states
        moves = torch.tensor(moves, dtype=torch._____)    # long for indices
        
        # Data augmentation (random subsequence) - OPTIONAL CHALLENGE
        if self.augment and len(states) > 10:
            start = np.random.randint(0, max(1, len(states) - 10))
            end = np.random.randint(start + 10, len(states) + 1)
            states = states[start:end]
            moves = moves[start:end]
        
        # TODO: Truncate if sequence is too long
        if self.max_length and len(states) > self.max_length:
            states = states[:_____]
            moves = moves[:_____]
        
        # TODO: Get the label index for this method
        label = self.method_to_idx[solve['_____']]
        
        return states, moves, label, len(states)

# Test the dataset
dataset = CubeTrajectoryDataset(data_path, max_length=170, augment=False)
print(f"\nDataset created with {len(dataset)} samples")

# Test getting a sample
states, moves, label, length = dataset[0]
print(f"Sample 0: states shape={states.shape}, moves shape={moves.shape}, label={label}, length={length}")
print(f"Method: {dataset.idx_to_method[label]}")

# EXPECTED OUTPUT:
# Loaded 2384 trajectories
# Methods: ['CFOP', 'LBL', 'Petrus', 'Roux', 'ZZ']
# Class weights: [~0.96, ~0.96, ~1.15, ~1.02, ~0.96]
# Sample 0: states shape=torch.Size([46, 324]), moves shape=torch.Size([46]), label=4, length=46


## Step 4: Custom Collate Function

**Theory**: Variable-length sequences can't be directly batched in PyTorch. We need a custom collate function to:
1. **Pad** sequences to the same length within each batch
2. Create **attention masks** to ignore padding positions
3. Return **lengths** for packed sequence processing

**Key Concept**: Padding allows batching, but we must track real vs. padded positions!


In [ ]:
# CODE CHALLENGE: Implement the collate function

def collate_fn(batch):
    """
    Custom collate function to handle variable-length sequences.
    
    HINT: Use pad_sequence from torch.nn.utils.rnn
    """
    # TODO: Unpack the batch into separate components
    states, moves, labels, lengths = _____
    
    # TODO: Pad sequences to the same length
    # Use pad_sequence with batch_first=True and padding_value=0
    states_padded = _____(states, batch_first=_____, padding_value=_____)
    moves_padded = _____(moves, batch_first=_____, padding_value=_____)
    
    # TODO: Convert labels and lengths to tensors
    labels = torch.tensor(labels, dtype=torch._____)
    lengths = torch.tensor(lengths, dtype=torch._____)
    
    # TODO: Create attention mask (1 for real values, 0 for padding)
    batch_size, max_len = states_padded.shape[:2]
    # This creates a mask where each position is True if it's within the sequence length
    mask = torch.arange(max_len).expand(batch_size, max_len) < lengths._____
    
    return states_padded, moves_padded, labels, lengths, mask

# Test the collate function with a small batch
test_loader = DataLoader(dataset, batch_size=3, shuffle=False, collate_fn=collate_fn)
batch = next(iter(test_loader))
states_batch, moves_batch, labels_batch, lengths_batch, mask_batch = batch

print("Batch shapes:")
print(f"States: {states_batch.shape}")
print(f"Moves: {moves_batch.shape}")
print(f"Labels: {labels_batch.shape}")
print(f"Lengths: {lengths_batch}")
print(f"Mask: {mask_batch.shape}")
print(f"Mask example:\n{mask_batch[0][:20]}")  # Show first 20 positions of first sample

# EXPECTED OUTPUT:
# States: torch.Size([3, max_len, 324])  # 3 samples, padded to max length
# Moves: torch.Size([3, max_len])
# Labels: torch.Size([3])
# Lengths: tensor([actual, lengths, here])
# Mask: torch.Size([3, max_len])
# Mask shows True for real positions, False for padding


## Step 5: Attention Mechanism

**Theory**: Attention allows the model to focus on the most important parts of the sequence. For cube solving:
- **CFOP**: Might focus on cross formation and last layer algorithms
- **Roux**: Might emphasize block building phases
- **LBL**: Might highlight layer transitions

**Implementation**: Compute attention scores → Apply softmax → Weight the sequence


In [ ]:
# CODE CHALLENGE: Implement the Attention Layer

class AttentionLayer(nn.Module):
    """
    Self-attention layer for sequence modeling.
    
    HINT: Attention = Linear layer → Mask → Softmax → Weighted sum
    """
    
    def __init__(self, hidden_size):
        super(AttentionLayer, self).__init__()
        self.hidden_size = hidden_size
        
        # TODO: Create a linear layer that maps from hidden_size to 1
        # This will compute attention scores for each position
        self.attention_weights = nn.Linear(_____, _____)
        
    def forward(self, lstm_output, mask=None):
        """
        Apply attention mechanism.
        
        Args:
            lstm_output: (batch, seq_len, hidden_size)
            mask: (batch, seq_len) boolean mask
        
        Returns:
            weighted_output: (batch, hidden_size)
            attention_weights: (batch, seq_len)
        """
        # TODO: Calculate attention scores
        # Apply the linear layer and squeeze the last dimension
        scores = self.attention_weights(lstm_output).squeeze(_____)  # (batch, seq_len)
        
        # TODO: Apply mask if provided
        # Set masked positions to a very negative value (-1e9)
        if mask is not None:
            scores = scores.masked_fill(~mask, _____)
        
        # TODO: Apply softmax to get attention weights
        attention_weights = F.softmax(scores, dim=_____)  # (batch, seq_len)
        
        # TODO: Apply attention weights using batch matrix multiplication
        # We want to multiply attention_weights with lstm_output
        weighted_output = torch.bmm(
            attention_weights.unsqueeze(_____),  # (batch, 1, seq_len)
            lstm_output  # (batch, seq_len, hidden_size)
        ).squeeze(_____)  # (batch, hidden_size)
        
        return weighted_output, attention_weights

# Test the attention layer
hidden_size = 256
attention = AttentionLayer(hidden_size)

# Create dummy LSTM output
batch_size, seq_len = 2, 10
dummy_lstm_out = torch.randn(batch_size, seq_len, hidden_size)
dummy_mask = torch.ones(batch_size, seq_len, dtype=torch.bool)
dummy_mask[0, 7:] = False  # Mask last 3 positions for first sample
dummy_mask[1, 5:] = False  # Mask last 5 positions for second sample

weighted_out, att_weights = attention(dummy_lstm_out, dummy_mask)
print(f"Attention output shape: {weighted_out.shape}")
print(f"Attention weights shape: {att_weights.shape}")
print(f"Attention weights sum (should be ~1): {att_weights.sum(dim=1)}")
print(f"Attention weights for sample 1: {att_weights[0].detach().numpy()[:10]}")

# EXPECTED OUTPUT:
# Attention output shape: torch.Size([2, 256])
# Attention weights shape: torch.Size([2, 10])
# Attention weights sum: tensor([1.0000, 1.0000]) (approximately)
# Attention weights: array of 10 values that sum to 1.0


## Checkpoint: What You've Learned So Far

**Congratulations!** You've implemented the core components:
- **Custom Dataset**: Handles variable-length sequences and class imbalance
- **Collate Function**: Pads sequences and creates attention masks  
- **Attention Layer**: Focuses on important parts of the sequence

## Step 6: Main LSTM Classifier Model

**Theory**: Now we combine everything into the main model:
1. **Move Embedding**: Convert discrete moves to learned representations
2. **Bidirectional LSTM**: Process sequences forward and backward
3. **Attention Mechanism**: Weight important sequence parts
4. **Multi-view Pooling**: Combine attention, max, and average pooling
5. **Classification Head**: Map features to method predictions

**Architecture Challenge**: This is the most complex part - take your time!


In [ ]:
# ADVANCED CODE CHALLENGE: Implement the Main LSTM Classifier

class CubeMethodClassifier(nn.Module):
    """
    Bidirectional LSTM with attention for classifying solving methods.
    
    Architecture:
    1. Move embedding layer (learns representations for each move type)
    2. Feature combination (concatenates state vector with move embedding)
    3. Bidirectional LSTM (2 layers)
    4. Attention mechanism
    5. Global pooling (max + average)
    6. Classification head with dropout
    """
    
    def __init__(self, 
                 state_dim=324,
                 num_moves=54,
                 move_embed_dim=32,
                 lstm_hidden=256,
                 lstm_layers=2,
                 num_classes=5,
                 dropout=0.3):
        """
        Initialize the classifier.
        
        Args:
            state_dim: Dimension of state vectors (324 for one-hot encoding)
            num_moves: Number of possible moves (54)
            move_embed_dim: Dimension of move embeddings
            lstm_hidden: Hidden size for LSTM
            lstm_layers: Number of LSTM layers
            num_classes: Number of methods to classify (5)
            dropout: Dropout probability
        """
        super(CubeMethodClassifier, self).__init__()
        
        # Move embedding layer
        # Learns a representation for each move type
        self.move_embedding = nn.Embedding(num_moves, move_embed_dim)
        
        # Input dimension is state + embedded move
        input_dim = state_dim + move_embed_dim
        
        # Bidirectional LSTM
        # Processes sequence in both directions to capture full context
        self.lstm = nn.LSTM(
            input_dim,
            lstm_hidden,
            lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0
        )
        
        # Attention layer
        # Identifies which parts of the sequence are most informative
        self.attention = AttentionLayer(lstm_hidden * 2)  # *2 for bidirectional
        
        # Classification head
        # Maps from LSTM features to class predictions
        self.dropout = nn.Dropout(dropout)
        
        # We concatenate attention output, max pool, and avg pool
        # This gives us multiple views of the sequence
        feature_dim = lstm_hidden * 2 * 3  # bidirectional * 3 pooling methods
        
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, lstm_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, lstm_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden // 2, num_classes)
        )
        
    def forward(self, states, moves, lengths, mask):
        """
        Forward pass through the network.
        
        Args:
            states: (batch, seq_len, 324) cube states
            moves: (batch, seq_len) move indices
            lengths: (batch,) actual sequence lengths
            mask: (batch, seq_len) attention mask
        
        Returns:
            logits: (batch, num_classes) classification scores
            attention_weights: (batch, seq_len) attention weights for visualization
        """
        batch_size = states.shape[0]
        
        # Embed moves
        # Convert discrete move indices to continuous representations
        move_embeds = self.move_embedding(moves)  # (batch, seq_len, embed_dim)
        
        # Combine state and move features
        # Each timestep now has both state information and move information
        combined = torch.cat([states, move_embeds], dim=-1)
        
        # Pack sequences for efficient LSTM processing
        # This handles variable-length sequences efficiently
        packed = pack_padded_sequence(
            combined, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        # Pass through LSTM
        # Bidirectional processing captures both forward and backward dependencies
        lstm_out, (hidden, cell) = self.lstm(packed)
        
        # Unpack sequences
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        
        # Apply attention mechanism
        # This identifies which parts of the solve are most characteristic
        attention_out, attention_weights = self.attention(lstm_out, mask)
        
        # Global max pooling
        # Captures the most distinctive features across the sequence
        masked_lstm = lstm_out.masked_fill(~mask.unsqueeze(-1), -1e9)
        max_pool = torch.max(masked_lstm, dim=1)[0]
        
        # Global average pooling
        # Captures the overall trajectory characteristics
        sum_pool = torch.sum(lstm_out * mask.unsqueeze(-1), dim=1)
        avg_pool = sum_pool / lengths.unsqueeze(-1).float()
        
        # Concatenate all features
        # Combines multiple views of the sequence for robust classification
        features = torch.cat([attention_out, max_pool, avg_pool], dim=-1)
        
        # Apply dropout and classify
        features = self.dropout(features)
        logits = self.classifier(features)
        
        return logits, attention_weights

# Test the model
model = CubeMethodClassifier(
    state_dim=324,
    num_moves=54,
    move_embed_dim=32,
    lstm_hidden=256,
    lstm_layers=2,
    num_classes=5,
    dropout=0.3
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test with a batch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Use the batch we created earlier
states_batch = states_batch.to(device)
moves_batch = moves_batch.to(device)
lengths_batch = lengths_batch.to(device)
mask_batch = mask_batch.to(device)

with torch.no_grad():
    logits, att_weights = model(states_batch, moves_batch, lengths_batch, mask_batch)
    
print(f"Output logits shape: {logits.shape}")
print(f"Attention weights shape: {att_weights.shape}")
print(f"Predicted classes: {torch.argmax(logits, dim=1)}")
print(f"Attention weights sum: {att_weights.sum(dim=1)}")


## Step 7: Training and Evaluation Functions

Now we'll implement the training loop and evaluation functions with proper metrics tracking.


In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    Train the model for one epoch.
    
    Returns:
        avg_loss: Average training loss
        accuracy: Training accuracy
    """
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm(dataloader, desc='Training')
    
    for states, moves, labels, lengths, mask in progress_bar:
        # Move to device
        states = states.to(device)
        moves = moves.to(device)
        labels = labels.to(device)
        lengths = lengths.to(device)
        mask = mask.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        logits, _ = model(states, moves, lengths, mask)
        
        # Calculate loss
        loss = criterion(logits, labels)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': loss.item(),
            'acc': 100. * correct / total
        })
    
    return total_loss / len(dataloader), 100. * correct / total


def evaluate(model, dataloader, criterion, device):
    """
    Evaluate the model on validation/test data.
    
    Returns:
        avg_loss: Average loss
        accuracy: Accuracy
        all_preds: All predictions for further analysis
        all_labels: All true labels
    """
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for states, moves, labels, lengths, mask in tqdm(dataloader, desc='Evaluating'):
            # Move to device
            states = states.to(device)
            moves = moves.to(device)
            labels = labels.to(device)
            lengths = lengths.to(device)
            mask = mask.to(device)
            
            # Forward pass
            logits, _ = model(states, moves, lengths, mask)
            
            # Calculate loss
            loss = criterion(logits, labels)
            
            # Track metrics
            total_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(dataloader), 100. * correct / total, all_preds, all_labels

print("Training and evaluation functions defined!")


## Step 8: Visualization Functions

These functions help us understand model performance and training progress.


In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, save_path='confusion_matrix.png'):
    """
    Plot and save confusion matrix.
    """
    cm = confusion_matrix(y_true, y_pred)
    
    # Normalize confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_normalized, annot=cm, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.ylabel('True Method')
    plt.xlabel('Predicted Method')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Confusion matrix saved to {save_path}")
    plt.show()


def plot_training_history(train_losses, val_losses, train_accs, val_accs, save_path='training_history.png'):
    """
    Plot training and validation curves.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Loss curves
    ax1.plot(train_losses, label='Train Loss')
    ax1.plot(val_losses, label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax2.plot(train_accs, label='Train Accuracy')
    ax2.plot(val_accs, label='Val Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Training history saved to {save_path}")
    plt.show()

print("Visualization functions defined!")


## Step 9: Complete Training Pipeline

Now let's put everything together in a complete training pipeline with proper data splitting, model initialization, and training loop.


In [ ]:
# Configuration
BATCH_SIZE = 32
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
MAX_LENGTH = 170  # Slightly above max observed length

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

# Create fresh dataset instance
dataset = CubeTrajectoryDataset(
    'dataset_generator/data/all_vector.json',
    max_length=MAX_LENGTH,
    augment=False
)

# Split into train/val/test
# We need indices for splitting
indices = list(range(len(dataset)))
labels = [dataset.data[i]['method'] for i in indices]

# Stratified split to maintain class balance
train_idx, test_idx = train_test_split(
    indices, test_size=0.3, stratify=labels, random_state=SEED
)

# Further split test into validation and test
test_labels = [labels[i] for i in test_idx]
val_idx, test_idx = train_test_split(
    test_idx, test_size=0.5, stratify=test_labels, random_state=SEED
)

print(f"\nDataset splits:")
print(f"  Train: {len(train_idx)} samples")
print(f"  Val: {len(val_idx)} samples")
print(f"  Test: {len(test_idx)} samples")

# Create data loaders
train_dataset = torch.utils.data.Subset(dataset, train_idx)
val_dataset = torch.utils.data.Subset(dataset, val_idx)
test_dataset = torch.utils.data.Subset(dataset, test_idx)

# Enable augmentation for training
dataset.augment = True

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)

# Disable augmentation for validation/test
dataset.augment = False

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)

print("Data loaders created successfully!")


In [ ]:
# Initialize model
model = CubeMethodClassifier(
    state_dim=324,
    num_moves=54,
    move_embed_dim=32,
    lstm_hidden=256,
    lstm_layers=2,
    num_classes=5,
    dropout=0.3
).to(device)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=dataset.class_weights.to(device))

# Optimizer with weight decay for regularization
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

print("Model, loss function, and optimizer initialized!")


## Step 10: Training Loop

Now let's run the complete training loop with validation and model checkpointing.


In [ ]:
# Training loop
print("\nStarting training...")
train_losses = []
val_losses = []
train_accs = []
val_accs = []
best_val_acc = 0

# Reduce epochs for demo (you can increase this for full training)
NUM_EPOCHS = 10  # Reduced for demonstration

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Learning rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Validate
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    print(f"\nTrain Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
        }, 'best_lstm_model.pth')
        print(f"Saved best model with validation accuracy: {val_acc:.2f}%")
    
    # Early stopping (simplified)
    if epoch > 5 and val_acc < max(val_accs[-5:]) - 5:
        print("Early stopping triggered")
        break

print("\nTraining completed!")


## Step 11: Evaluation and Analysis

Let's evaluate our trained model and analyze the results.


In [ ]:
# Plot training history
plot_training_history(train_losses, val_losses, train_accs, val_accs)

# Load best model for final evaluation
print("\nLoading best model for final evaluation...")
checkpoint = torch.load('best_lstm_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])

# Final test evaluation
print("\nEvaluating on test set...")
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.2f}%")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(
    test_labels, test_preds,
    target_names=dataset.methods,
    digits=3
))

# Plot confusion matrix
plot_confusion_matrix(test_labels, test_preds, dataset.methods)


## Step 12: Analyze Performance by Method and Sequence Length

Let's dive deeper into understanding which methods are easier/harder to classify and how sequence length affects performance.


In [ ]:
# Analyze errors by sequence length and method
print("Analyzing performance by sequence length and method...")
test_lengths = []
test_methods = []
test_correct = []

for idx in test_idx:
    solve = dataset.data[idx]
    test_lengths.append(len(solve['steps']))
    test_methods.append(solve['method'])

# Convert predictions to method names for analysis
test_pred_methods = [dataset.idx_to_method[pred] for pred in test_preds]
test_true_methods = [dataset.idx_to_method[label] for label in test_labels]
test_correct = [pred == true for pred, true in zip(test_pred_methods, test_true_methods)]

# Group by method and analyze
print(f"\nPer-method performance:")
print(f"{'Method':<8} {'Accuracy':<10} {'Avg Length':<12} {'Count':<8}")
print("-" * 40)

for method in dataset.methods:
    method_mask = [m == method for m in test_true_methods]
    if sum(method_mask) > 0:
        method_correct = [test_correct[i] for i in range(len(test_correct)) if method_mask[i]]
        method_lengths = [test_lengths[i] for i in range(len(test_lengths)) if method_mask[i]]
        method_acc = sum(method_correct) / len(method_correct) * 100
        avg_length = sum(method_lengths) / len(method_lengths)
        count = len(method_correct)
        print(f"{method:<8} {method_acc:<10.1f}% {avg_length:<12.1f} {count:<8}")

# Analyze accuracy vs sequence length
print(f"\nAccuracy by sequence length ranges:")
length_ranges = [(0, 50), (50, 75), (75, 100), (100, 125), (125, 200)]

for min_len, max_len in length_ranges:
    range_mask = [(min_len <= length < max_len) for length in test_lengths]
    if sum(range_mask) > 0:
        range_correct = [test_correct[i] for i in range(len(test_correct)) if range_mask[i]]
        range_acc = sum(range_correct) / len(range_correct) * 100
        count = len(range_correct)
        print(f"Length {min_len}-{max_len}: {range_acc:.1f}% accuracy ({count} samples)")


## Step 13: Attention Visualization

Let's visualize the attention weights to understand what parts of the solving sequence the model focuses on.


In [ ]:
def visualize_attention(model, dataset, sample_idx, device, max_steps=50):
    """
    Visualize attention weights for a specific sample.
    """
    # Get the sample
    states, moves, label, length = dataset[sample_idx]
    method_name = dataset.idx_to_method[label]
    
    # Prepare batch (single sample)
    states_batch = states.unsqueeze(0).to(device)
    moves_batch = moves.unsqueeze(0).to(device)
    lengths_batch = torch.tensor([length]).to(device)
    mask_batch = torch.ones(1, length, dtype=torch.bool).to(device)
    
    # Get predictions and attention weights
    model.eval()
    with torch.no_grad():
        logits, attention_weights = model(states_batch, moves_batch, lengths_batch, mask_batch)
        predicted_class = torch.argmax(logits, dim=1).item()
        predicted_method = dataset.idx_to_method[predicted_class]
    
    # Plot attention weights
    attention = attention_weights[0][:length].cpu().numpy()
    
    # Limit visualization to max_steps for readability
    plot_length = min(length, max_steps)
    
    plt.figure(figsize=(15, 6))
    
    # Plot attention weights
    plt.subplot(1, 2, 1)
    plt.plot(range(plot_length), attention[:plot_length], 'b-', linewidth=2)
    plt.fill_between(range(plot_length), attention[:plot_length], alpha=0.3)
    plt.title(f'Attention Weights\nTrue: {method_name}, Predicted: {predicted_method}')
    plt.xlabel('Step in Sequence')
    plt.ylabel('Attention Weight')
    plt.grid(True, alpha=0.3)
    
    # Plot move sequence (first few moves)
    plt.subplot(1, 2, 2)
    move_indices = moves[:plot_length].numpy()
    plt.bar(range(plot_length), move_indices, alpha=0.7)
    plt.title('Move Sequence (Move Indices)')
    plt.xlabel('Step in Sequence')
    plt.ylabel('Move Index')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print top attention steps
    top_attention_indices = np.argsort(attention)[-5:][::-1]
    print(f"\nTop 5 most attended steps:")
    for i, step_idx in enumerate(top_attention_indices):
        move_idx = moves[step_idx].item()
        att_weight = attention[step_idx]
        print(f"{i+1}. Step {step_idx}: Move {move_idx}, Attention: {att_weight:.4f}")

# Visualize attention for a few samples from different methods
print("Visualizing attention for sample trajectories...")

# Find samples from different methods
method_samples = {}
for idx, solve in enumerate(dataset.data):
    method = solve['method']
    if method not in method_samples:
        method_samples[method] = []
    if len(method_samples[method]) < 2:  # Get 2 samples per method
        method_samples[method].append(idx)

# Visualize one sample from each method
for method, sample_indices in method_samples.items():
    if sample_indices:
        print(f"\n{'='*50}")
        print(f"Attention visualization for {method} method:")
        visualize_attention(model, dataset, sample_indices[0], device)


## Step 14: Model Saving and Loading

Finally, let's save our trained model with all necessary information for future use.


In [ ]:
# Save final model with complete configuration
final_model_path = 'final_lstm_model.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'dataset_info': {
        'methods': dataset.methods,
        'method_to_idx': dataset.method_to_idx,
        'idx_to_method': dataset.idx_to_method,
        'class_weights': dataset.class_weights.numpy().tolist()
    },
    'config': {
        'state_dim': 324,
        'num_moves': 54,
        'move_embed_dim': 32,
        'lstm_hidden': 256,
        'lstm_layers': 2,
        'num_classes': 5,
        'dropout': 0.3,
        'max_length': MAX_LENGTH
    },
    'performance': {
        'test_accuracy': test_acc,
        'test_loss': test_loss,
        'best_val_accuracy': best_val_acc
    },
    'training_info': {
        'epochs_trained': len(train_losses),
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE
    }
}, final_model_path)

print(f"Final model saved to {final_model_path}")

# Demonstrate how to load the model
print("\nDemonstrating model loading...")

def load_trained_model(model_path, device):
    """
    Load a trained model from checkpoint.
    """
    checkpoint = torch.load(model_path, map_location=device)
    
    # Recreate model with saved configuration
    config = checkpoint['config']
    model = CubeMethodClassifier(
        state_dim=config['state_dim'],
        num_moves=config['num_moves'],
        move_embed_dim=config['move_embed_dim'],
        lstm_hidden=config['lstm_hidden'],
        lstm_layers=config['lstm_layers'],
        num_classes=config['num_classes'],
        dropout=config['dropout']
    ).to(device)
    
    # Load trained weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    return model, checkpoint

# Test loading
loaded_model, checkpoint_info = load_trained_model(final_model_path, device)
print(f"Model loaded successfully!")
print(f"Test accuracy: {checkpoint_info['performance']['test_accuracy']:.2f}%")
print(f"Available methods: {checkpoint_info['dataset_info']['methods']}")


## Tutorial Complete: What You've Accomplished

**Congratulations!** You've learned the fundamentals of sequence classification with LSTMs:

### Key Skills Mastered
1. **Custom PyTorch Datasets** for complex, variable-length data
2. **Attention Mechanisms** for focusing on important sequence parts  
3. **Variable-length Sequence Handling** with padding and masking
4. **Class Imbalance Solutions** using weighted loss functions

### Next Steps to Complete the Full Implementation

The remaining components follow similar patterns:

#### **Main LSTM Model** (Step 6)
- Combine move embeddings + bidirectional LSTM + attention
- Use `nn.Embedding()` for moves, `nn.LSTM(bidirectional=True)` for sequences
- Concatenate attention, max pooling, and average pooling features

#### **Training Functions** (Steps 7-10)  
- Standard PyTorch training loop with gradient clipping
- Validation with early stopping and model checkpointing
- Use `torch.nn.utils.clip_grad_norm_()` to prevent exploding gradients

#### **Evaluation & Visualization** (Steps 11-14)
- Classification metrics and confusion matrices
- Attention weight visualization to understand model focus
- Performance analysis by method and sequence length

### Reference Implementation

Check `/n/home04/amuppidi/Rubik-s-AI/lstm.ipynb` for the complete solution with:
- Full model implementation (~3.5M parameters)
- Complete training pipeline with GPU support
- Comprehensive evaluation and visualization
- Expected 85-95% accuracy on test data

### Challenge Yourself

Try implementing the remaining parts yourself first, then compare with the solution!

**Key Architecture Decisions to Remember**:
- **Bidirectional LSTM**: Captures both forward/backward dependencies
- **Multi-view Pooling**: Combines attention + max + average for robustness  
- **Packed Sequences**: Efficient processing of variable-length data
- **Gradient Clipping**: Prevents exploding gradients in RNNs

### Applications

Your LSTM classifier can be extended for:
- **Real-time Method Detection** during speedcubing competitions
- **Training Analysis** to help cubers improve their techniques
- **Educational Tools** for teaching different solving approaches
- **Algorithm Recommendation** based on scramble patterns

**Great work on completing this advanced deep learning tutorial!**
